# Silver Layer — Spark Structured Streaming

Reads Delta tables from the **Bronze Lakehouse** (Mirrored DB via OneLake Shortcut)  
and writes continuously to the **Silver Lakehouse** with enrichment and cleansing.

## What this notebook does
1. Reads `SensorReadings` stream from Bronze (Delta source)
2. Joins static lookup tables (`Sensors`, `ProcessUnits`) to enrich each reading
3. Casts types, adds derived columns (`ingest_ts`, `is_in_alarm`, `deviation_pct`)
4. Writes to Silver in **append** mode with `processingTime = "30 seconds"`

## Prerequisites
- Bronze Lakehouse with Mirrored DB shortcuts configured (ADR-001 pattern)
- Silver Lakehouse created (empty)
- Spark pool: enable **Spark Always-On** or keep this notebook session active
- Set the two notebook parameters below before running

## Imports notebook into Fabric
Workspace → + New → Import Notebook → select this `.ipynb` file

In [ ]:
# ── Parameters — set these before running ─────────────────────────────────────
# Replace with your actual Lakehouse names (or ABFS paths)
BRONZE_LAKEHOUSE = "bronze_lakehouse"   # Fabric Lakehouse display name
SILVER_LAKEHOUSE = "silver_lakehouse"   # Fabric Lakehouse display name

# Streaming trigger interval
PROCESSING_TIME  = "30 seconds"

# Checkpoint root in Silver Lakehouse Files/ section
CHECKPOINT_ROOT  = "Files/checkpoints/silver_streaming"

In [ ]:
# ── Resolve Lakehouse paths via Fabric notebookutils ──────────────────────────
from notebookutils import mssparkutils

bronze_path = mssparkutils.lakehouse.getProperties(BRONZE_LAKEHOUSE).abfsPath
silver_path = mssparkutils.lakehouse.getProperties(SILVER_LAKEHOUSE).abfsPath

bronze_tables = f"{bronze_path}/Tables"
silver_tables = f"{silver_path}/Tables"
checkpoint_base = f"{silver_path}/{CHECKPOINT_ROOT}"

print(f"Bronze tables : {bronze_tables}")
print(f"Silver tables : {silver_tables}")
print(f"Checkpoint    : {checkpoint_base}")

In [ ]:
# ── Load static reference tables (batch — small, rarely change) ───────────────
from pyspark.sql import functions as F

df_sensors = (
    spark.read.format("delta")
    .load(f"{bronze_tables}/Sensors")
    .select(
        "sensor_id", "tag_id", "tag_name",
        "equipment_id", "parameter_type", "unit_of_measure",
        "normal_min", "normal_max", "alarm_low", "alarm_high"
    )
).cache()

df_units = (
    spark.read.format("delta")
    .load(f"{bronze_tables}/ProcessUnits")
    .select("unit_id", "unit_name", "unit_type", "parent_unit_id")
).cache()

print(f"Sensors loaded  : {df_sensors.count()} rows")
print(f"Process units   : {df_units.count()} rows")

In [ ]:
# ── Read SensorReadings as a streaming Delta source ───────────────────────────
df_stream = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")
    .load(f"{bronze_tables}/SensorReadings")
)

print("Streaming schema:")
df_stream.printSchema()

In [ ]:
# ── Enrich: join sensors + units, derive status columns ──────────────────────
df_enriched = (
    df_stream
    # Cast timestamp (Mirrored DB lands it as string in some schema versions)
    .withColumn("ts",         F.col("ts").cast("timestamp"))
    .withColumn("value",      F.col("value").cast("double"))
    .withColumn("quality",    F.col("quality").cast("integer"))
    # Enrich with sensor catalog
    .join(F.broadcast(df_sensors), on="sensor_id", how="inner")
    # Enrich with equipment / train name
    .join(
        F.broadcast(df_units.withColumnRenamed("unit_id", "equipment_id")
                            .withColumnRenamed("unit_name", "equipment_name")
                            .withColumnRenamed("unit_type", "equipment_type")),
        on="equipment_id",
        how="left"
    )
    # Derived: deviation from midpoint as a percentage of normal range
    .withColumn(
        "deviation_pct",
        F.round(
            100.0 * (F.col("value") - (F.col("normal_min") + F.col("normal_max")) / 2.0)
            / ((F.col("normal_max") - F.col("normal_min")) / 2.0),
            2
        )
    )
    # Derived: is this reading outside the normal band?
    .withColumn(
        "is_out_of_range",
        (F.col("value") > F.col("normal_max")) | (F.col("value") < F.col("normal_min"))
    )
    # Derived: alarm status (ISA-18.2 naming)
    .withColumn(
        "alarm_status",
        F.when(F.col("alarm_high").isNotNull() & (F.col("value") > F.col("alarm_high")), "ALARM-H")
        .when(F.col("alarm_low").isNotNull()  & (F.col("value") < F.col("alarm_low")),  "ALARM-L")
        .when(F.col("value") > F.col("normal_max"),                                     "HIGH")
        .when(F.col("value") < F.col("normal_min"),                                     "LOW")
        .otherwise("NORMAL")
    )
    # Metadata
    .withColumn("ingest_ts", F.current_timestamp())
    # Select final Silver schema
    .select(
        "reading_id", "sensor_id", "ts", "value", "quality",
        "tag_id", "tag_name", "parameter_type", "unit_of_measure",
        "equipment_id", "equipment_name", "equipment_type",
        "normal_min", "normal_max", "alarm_low", "alarm_high",
        "deviation_pct", "is_out_of_range", "alarm_status",
        "ingest_ts"
    )
)

In [ ]:
# ── Write to Silver Lakehouse — always-on streaming ───────────────────────────
# processingTime triggers a micro-batch every 30 seconds.
# checkpointLocation ensures exactly-once semantics on restart.
query = (
    df_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{checkpoint_base}/SensorReadings")
    .trigger(processingTime=PROCESSING_TIME)
    .start(f"{silver_tables}/SensorReadings")
)

print(f"Streaming query started: {query.id}")
print("Status:", query.status)

# Keep the notebook session alive — required for always-on streaming.
# To stop: query.stop()
query.awaitTermination()